# Notebook 09 — LSTM and GRU

## Purpose

This notebook introduces two sequence models:

- **LSTM:** a recurrent neural network with a more detailed memory mechanism;
- **GRU:** a simpler recurrent network with fewer gates and usually fewer parameters.

They receive a sequence of past returns and predict the next return.

This notebook first builds a clear static train/validation/test experiment. After it works, the training function can be called from the walk-forward engine.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Find the project root whether the notebook is launched from:
#   project/
# or:
#   project/notebooks/
CURRENT = Path.cwd().resolve()

if (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif CURRENT.name == "notebooks" and (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    possible_roots = [CURRENT, *CURRENT.parents]
    matches = [p for p in possible_roots if (p / "data").exists() and (p / "notebooks").exists()]
    if not matches:
        raise FileNotFoundError(
            "Could not find the project root. Open the sp500-forecasting-dissertation "
            "folder in VS Code, then run this notebook again."
        )
    PROJECT_ROOT = matches[0]

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "reports" / "models"

for folder in [DATA_PROCESSED, REPORT_TABLES, REPORT_FIGURES, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: <project_root>
Python: /opt/anaconda3/envs/dissertation/bin/python
Python version: 3.11.15


In [2]:
def find_crsp_panel(processed_folder: Path) -> Path:
    """Find the best available corrected CRSP parquet file."""
    preferred_names = [
        "crsp_sp500_daily_corrected_2010_2024.parquet",
        "crsp_sp500_daily_corrected.parquet",
        "crsp_daily_corrected_2010_2024.parquet",
        "crsp_daily_2010_2024.parquet",
    ]

    for name in preferred_names:
        path = processed_folder / name
        if path.exists():
            return path

    candidates = sorted(
        processed_folder.glob("*.parquet"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No parquet file was found in {processed_folder}. "
            "Run and save the corrected CRSP data notebook first."
        )

    print("No preferred filename found. Using the newest parquet file:")
    return candidates[0]


PANEL_PATH = find_crsp_panel(DATA_PROCESSED)
panel = pd.read_parquet(PANEL_PATH).copy()
panel.columns = [str(c).strip().lower() for c in panel.columns]

# Accept the possible names used in the earlier audit notebook.
return_candidates = ["ret", "combined_return", "ret_combined"]
return_source = next((c for c in return_candidates if c in panel.columns), None)

required_base = {"permno", "date"}
missing_base = required_base.difference(panel.columns)

if missing_base:
    raise ValueError(f"Missing required columns: {sorted(missing_base)}")

if return_source is None:
    raise ValueError(
        "Could not find a return column. Expected one of: "
        f"{return_candidates}. Found: {panel.columns.tolist()}"
    )

panel["date"] = pd.to_datetime(panel["date"])
panel["permno"] = pd.to_numeric(panel["permno"], errors="coerce").astype("Int64")
panel[return_source] = pd.to_numeric(panel[return_source], errors="coerce")

if "mktcap" in panel.columns:
    panel["mktcap"] = pd.to_numeric(panel["mktcap"], errors="coerce")

panel = (
    panel.dropna(subset=["permno", "date"])
         .sort_values(["date", "permno"])
         .reset_index(drop=True)
)

# IMPORTANT DECISION:
# "simple" keeps CRSP total returns, including a possible -100% delisting return.
# "log" uses log(1 + return), but an exact -100% return cannot be logged.
RETURN_MODE = "simple"

if RETURN_MODE == "simple":
    panel["model_return"] = panel[return_source]
elif RETURN_MODE == "log":
    impossible_for_log = panel[return_source] <= -1
    print("Rows not usable as log-returns:", int(impossible_for_log.sum()))
    panel["model_return"] = np.where(
        panel[return_source] > -1,
        np.log1p(panel[return_source]),
        np.nan,
    )
else:
    raise ValueError("RETURN_MODE must be 'simple' or 'log'.")

print("Loaded:", PANEL_PATH)
print("Rows:", len(panel))
print("Dates:", panel["date"].min(), "to", panel["date"].max())
print("Unique PERMNOs:", panel["permno"].nunique())
print("Return source:", return_source)
print("Return mode:", RETURN_MODE)
print("Missing modelling returns:", panel["model_return"].isna().sum())

Loaded: <project_root>/data/processed/crsp_sp500_daily_corrected_2010_2024.parquet
Rows: 1711517
Dates: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Unique PERMNOs: 740
Return source: ret
Return mode: simple
Missing modelling returns: 60


In [3]:
def latest_cross_section_before(data: pd.DataFrame, as_of_date) -> pd.DataFrame:
    """
    Return the cross-section on the latest trading date on or before as_of_date.

    We deliberately do NOT take each stock's individually latest historical row.
    Doing that could accidentally include a company that left the S&P 500 years ago.
    Using one common market date keeps only securities present in the corrected panel
    on that date.
    """
    as_of_date = pd.Timestamp(as_of_date)

    eligible_dates = data.loc[
        data["date"] <= as_of_date,
        "date"
    ]

    if eligible_dates.empty:
        raise ValueError(f"No data available on or before {as_of_date.date()}.")

    market_date = eligible_dates.max()

    cross_section = data.loc[
        data["date"] == market_date
    ].copy()

    if cross_section.empty:
        raise ValueError(f"No cross-section found for {market_date.date()}.")

    return cross_section


def select_top_n_by_lagged_market_cap(
    data: pd.DataFrame,
    as_of_date,
    n_stocks: int,
) -> list[int]:
    """
    Select stocks using market capitalisation from one common historical date.
    """
    if "mktcap" not in data.columns:
        raise ValueError("The panel needs a 'mktcap' column for top-N selection.")

    cross_section = latest_cross_section_before(data, as_of_date)
    cross_section = cross_section.dropna(subset=["mktcap"])
    cross_section = cross_section[cross_section["mktcap"] > 0]

    selected = (
        cross_section.nlargest(n_stocks, "mktcap")["permno"]
                     .astype(int)
                     .tolist()
    )

    if len(selected) < n_stocks:
        print(f"Warning: requested {n_stocks} stocks, but selected {len(selected)}.")

    return selected

## Install requirement

This notebook requires PyTorch.

Run in the VS Code terminal once:

```bash
python -m pip install torch scikit-learn
```

In [4]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)

Device: mps
PyTorch version: 2.12.1


## Quick-mode settings

The dissertation specification may use 252 past trading days and many stocks.

For learning and debugging, start smaller. Increase the settings only after the notebook works.

In [5]:
QUICK_MODE = True

TRAIN_END = pd.Timestamp("2018-12-31")
VALIDATION_END = pd.Timestamp("2019-12-31")

if QUICK_MODE:
    N_STOCKS = 10
    LOOKBACK = 60
    HIDDEN_SIZE = 32
    EPOCHS = 5
    BATCH_SIZE = 128
else:
    N_STOCKS = 100
    LOOKBACK = 252
    HIDDEN_SIZE = 64
    EPOCHS = 50
    BATCH_SIZE = 256

PATIENCE = 5
LEARNING_RATE = 1e-3

print({
    "n_stocks": N_STOCKS,
    "lookback": LOOKBACK,
    "hidden_size": HIDDEN_SIZE,
    "epochs": EPOCHS,
})

{'n_stocks': 10, 'lookback': 60, 'hidden_size': 32, 'epochs': 5}


## Select stocks without using future market-cap information

The universe is chosen using market capitalisation available by the end of 2018.

This avoids selecting stocks using knowledge from the future test period.

In [6]:
selected_permnos = select_top_n_by_lagged_market_cap(
    panel,
    as_of_date=TRAIN_END,
    n_stocks=N_STOCKS,
)

model_panel = panel[panel["permno"].isin(selected_permnos)].copy()

wide_returns = (
    model_panel.pivot_table(
        index="date",
        columns="permno",
        values="model_return",
        aggfunc="first",
    )
    .sort_index()
)

print("Wide panel shape:", wide_returns.shape)
print("Selected PERMNOs:", selected_permnos)

Wide panel shape: (3774, 10)
Selected PERMNOs: [10107, 14593, 84788, 14542, 22111, 47896, 13407, 90319, 11850, 83443]


## Standardise using training data only

Neural networks train more easily when inputs have a similar scale.

To prevent leakage:

- calculate each stock's mean and standard deviation using only 2010–2018;
- use those same values to transform validation and test data.

In [7]:
train_wide = wide_returns.loc[:TRAIN_END]

train_means = train_wide.mean()
train_stds = train_wide.std().replace(0, np.nan)

standardised = (wide_returns - train_means) / train_stds

print("Training means after scaling:")
print(standardised.loc[:TRAIN_END].mean().round(3).head())

print("\nTraining standard deviations after scaling:")
print(standardised.loc[:TRAIN_END].std().round(3).head())

Training means after scaling:
permno
10107    0.0
11850   -0.0
13407   -0.0
14542   -0.0
14593   -0.0
dtype: Float64

Training standard deviations after scaling:
permno
10107    1.0
11850    1.0
13407    1.0
14542    1.0
14593    1.0
dtype: Float64


## Build sequences

For every stock:

- input `X`: the previous `LOOKBACK` returns;
- target `y`: the next return;
- target date decides whether the sample belongs to train, validation or test.

Missing sequences are skipped rather than filled with invented returns.

In [8]:
def create_sequence_records(
    standardised_panel: pd.DataFrame,
    raw_panel: pd.DataFrame,
    lookback: int,
):
    records = []

    dates = standardised_panel.index.to_numpy()

    for permno in standardised_panel.columns:
        x_values = standardised_panel[permno].to_numpy(dtype=float)
        y_values = raw_panel[permno].to_numpy(dtype=float)

        for i in range(lookback, len(dates)):
            x_window = x_values[i - lookback:i]
            target_standardised = x_values[i]
            target_raw = y_values[i]

            if (
                np.isnan(x_window).any()
                or np.isnan(target_standardised)
                or np.isnan(target_raw)
            ):
                continue

            records.append({
                "x": x_window.astype(np.float32),
                "y_standardised": np.float32(target_standardised),
                "y_raw": np.float32(target_raw),
                "date": pd.Timestamp(dates[i]),
                "permno": int(permno),
            })

    return records


records = create_sequence_records(
    standardised,
    wide_returns,
    lookback=LOOKBACK,
)

print("Total valid sequences:", len(records))

Total valid sequences: 35041


In [9]:
train_records = [r for r in records if r["date"] <= TRAIN_END]
validation_records = [
    r for r in records
    if TRAIN_END < r["date"] <= VALIDATION_END
]
test_records = [r for r in records if r["date"] > VALIDATION_END]

print("Train:", len(train_records))
print("Validation:", len(validation_records))
print("Test:", len(test_records))

Train: 19941
Validation: 2520
Test: 12580


## Dataset and DataLoader

A Dataset stores samples.

A DataLoader groups them into batches so the network does not process every sequence at once.

In [10]:
class ReturnSequenceDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        x = torch.tensor(record["x"], dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(record["y_standardised"], dtype=torch.float32)
        permno = torch.tensor(record["permno"], dtype=torch.long)
        y_raw = torch.tensor(record["y_raw"], dtype=torch.float32)

        # Dates are intentionally not returned through DataLoader because
        # pandas.Timestamp objects are not handled by the default PyTorch collator.
        # We reattach dates later using the original ordered test_records list.
        return x, y, permno, y_raw


train_loader = DataLoader(
    ReturnSequenceDataset(train_records),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

validation_loader = DataLoader(
    ReturnSequenceDataset(validation_records),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    ReturnSequenceDataset(test_records),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

## Define one model class for both LSTM and GRU

The only difference is which recurrent cell is selected.

Input shape:

`batch × sequence length × number of features`

In [11]:
class SequenceRegressor(nn.Module):
    def __init__(
        self,
        cell_type: str,
        hidden_size: int = 32,
        num_layers: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()

        cell_type = cell_type.upper()

        recurrent_class = {
            "LSTM": nn.LSTM,
            "GRU": nn.GRU,
        }.get(cell_type)

        if recurrent_class is None:
            raise ValueError("cell_type must be 'LSTM' or 'GRU'.")

        recurrent_dropout = dropout if num_layers > 1 else 0.0

        self.recurrent = recurrent_class(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence_output, _ = self.recurrent(x)
        last_output = sequence_output[:, -1, :]
        return self.output(self.dropout(last_output)).squeeze(-1)

## Train with early stopping

Early stopping prevents the model from continuing after validation performance stops improving.

In [12]:
def evaluate_loss(model, loader, criterion):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y, _, _ in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            prediction = model(x)
            loss = criterion(prediction, y)
            losses.append(loss.item())

    return float(np.mean(losses)) if losses else np.nan


def train_sequence_model(cell_type: str):
    model = SequenceRegressor(
        cell_type=cell_type,
        hidden_size=HIDDEN_SIZE,
        num_layers=1,
        dropout=0.2,
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-5,
    )

    best_validation_loss = np.inf
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        training_losses = []

        for x, y, _, _ in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimiser.zero_grad()
            prediction = model(x)
            loss = criterion(prediction, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()

            training_losses.append(loss.item())

        train_loss = float(np.mean(training_losses))
        validation_loss = evaluate_loss(model, validation_loader, criterion)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
        })

        print(
            f"{cell_type} epoch {epoch:02d} | "
            f"train {train_loss:.6f} | "
            f"validation {validation_loss:.6f}"
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [13]:
lstm_model, lstm_history = train_sequence_model("LSTM")
gru_model, gru_history = train_sequence_model("GRU")

LSTM epoch 01 | train 0.997626 | validation 0.846616
LSTM epoch 02 | train 0.996081 | validation 0.846501
LSTM epoch 03 | train 0.994438 | validation 0.845695
LSTM epoch 04 | train 0.992755 | validation 0.848830
LSTM epoch 05 | train 0.992080 | validation 0.849974
GRU epoch 01 | train 0.998525 | validation 0.845797
GRU epoch 02 | train 0.996648 | validation 0.845603
GRU epoch 03 | train 0.995921 | validation 0.845304
GRU epoch 04 | train 0.996144 | validation 0.845070
GRU epoch 05 | train 0.995368 | validation 0.845641


## Create test predictions and return to the original scale

The networks predict standardised returns.

For interpretation and evaluation, each prediction is converted back using that stock's training mean and standard deviation.

In [14]:
def predict_sequence_model(model, loader, model_name: str):
    model.eval()
    rows = []

    with torch.no_grad():
        for x, y_standardised, permnos, actual_raw in loader:
            prediction_standardised = model(x.to(DEVICE)).cpu().numpy()

            permnos = permnos.cpu().numpy().astype(int)
            actual_raw = actual_raw.cpu().numpy().astype(float)

            for pred_std, permno, actual in zip(
                prediction_standardised,
                permnos,
                actual_raw,
            ):
                stock_mean = float(train_means.loc[permno])
                stock_std = float(train_stds.loc[permno])
                forecast_raw = pred_std * stock_std + stock_mean

                rows.append({
                    "permno": int(permno),
                    "model": model_name,
                    "forecast": float(forecast_raw),
                    "actual": float(actual),
                })

    return pd.DataFrame(rows)


lstm_predictions_without_dates = predict_sequence_model(
    lstm_model,
    test_loader,
    "lstm",
)

gru_predictions_without_dates = predict_sequence_model(
    gru_model,
    test_loader,
    "gru",
)

# Test DataLoader has shuffle=False, so its output order matches test_records.
test_index = pd.DataFrame({
    "date": [record["date"] for record in test_records],
    "permno_check": [record["permno"] for record in test_records],
})

lstm_predictions = pd.concat(
    [test_index.reset_index(drop=True), lstm_predictions_without_dates],
    axis=1,
)

gru_predictions = pd.concat(
    [test_index.reset_index(drop=True), gru_predictions_without_dates],
    axis=1,
)

assert (
    lstm_predictions["permno"].to_numpy()
    == lstm_predictions["permno_check"].to_numpy()
).all()

assert (
    gru_predictions["permno"].to_numpy()
    == gru_predictions["permno_check"].to_numpy()
).all()

lstm_predictions = lstm_predictions.drop(columns="permno_check")
gru_predictions = gru_predictions.drop(columns="permno_check")

deep_predictions = pd.concat(
    [lstm_predictions, gru_predictions],
    ignore_index=True,
)

deep_predictions.head()

,date,permno,model,forecast,actual
0,2020-01-02,10107,lstm,0.000499,0.018516
1,2020-01-03,10107,lstm,0.000282,-0.012452
2,2020-01-06,10107,lstm,0.000422,0.002585
3,2020-01-07,10107,lstm,0.000481,-0.009118
4,2020-01-08,10107,lstm,0.000715,0.015928


In [15]:
def prediction_metrics(group):
    error = group["actual"] - group["forecast"]

    return pd.Series({
        "observations": len(group),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "mae": np.mean(np.abs(error)),
        "directional_accuracy": np.mean(
            np.sign(group["actual"]) == np.sign(group["forecast"])
        ),
    })


deep_metrics = (
    deep_predictions.groupby("model")
                    .apply(prediction_metrics)
                    .reset_index()
)

deep_metrics

,model,observations,rmse,mae,directional_accuracy
0,gru,12580.0,0.020281,0.013772,0.527027
1,lstm,12580.0,0.020297,0.013779,0.524881


In [16]:
deep_predictions.to_parquet(
    DATA_PROCESSED / "lstm_gru_predictions.parquet",
    index=False,
)

deep_metrics.to_csv(
    REPORT_TABLES / "lstm_gru_metrics.csv",
    index=False,
)

torch.save(lstm_model.state_dict(), MODEL_DIR / "lstm_state_dict.pt")
torch.save(gru_model.state_dict(), MODEL_DIR / "gru_state_dict.pt")

lstm_history.to_csv(REPORT_TABLES / "lstm_training_history.csv", index=False)
gru_history.to_csv(REPORT_TABLES / "gru_training_history.csv", index=False)

print("Saved LSTM and GRU outputs.")

Saved LSTM and GRU outputs.


## Limitation of this notebook

This first experiment trains each deep model once.

The final dissertation should connect the training functions to the walk-forward framework so that model updates follow the same historical schedule as the classical baselines.

Do not claim the static results are the final walk-forward results.

## Walk-forward retraining (run this on your own machine, not in a constrained sandbox)

Everything above this point is the **static** fit: train once through 2018, predict straight through 2020-2024. `README_NEXT_NOTEBOOKS.md` flags this explicitly as not equivalent to walk-forward evaluation.

The cell below uses `src/evaluation/walk_forward_dl.py` to retrain every 21 trading days, matching notebook 08's classical walk-forward design. It warm-starts from the previous block's weights and fine-tunes for a handful of epochs per block rather than refitting from scratch each time — full from-scratch refits ~60 times over would multiply an already multi-hour single fit by 60. This is documented in the module docstring; describe it the same way in your methodology section.

**Timing reality (measured directly on a single CPU core):** ~4.3 seconds per training batch at full scale. A full from-scratch fit at `N_STOCKS=100, LOOKBACK=252, EPOCHS=50` is on the order of tens of hours; this warm-started version is far cheaper but still real compute — budget at least a few hours, and prefer a machine with more cores or a GPU if available.

In [ ]:
RUN_WALK_FORWARD_DL = False  # set True once QUICK_MODE is also False, and you're running this locally

if RUN_WALK_FORWARD_DL:
    import sys as _sys
    if str(PROJECT_ROOT) not in _sys.path:
        _sys.path.insert(0, str(PROJECT_ROOT))
    from src.evaluation.walk_forward_dl import WalkForwardDLConfig, walk_forward_train_predict

    wf_config_template = dict(
        lookback=LOOKBACK,
        hidden_size=HIDDEN_SIZE,
        initial_epochs=EPOCHS,
        finetune_epochs=5,       # epochs per subsequent retrain block (see markdown above)
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        retrain_every=21,
        device=DEVICE,
    )

    walk_forward_predictions = []
    for cell_type in ["LSTM", "GRU"]:
        print(f"--- walk-forward {cell_type} ---")
        config = WalkForwardDLConfig(cell_type=cell_type, **wf_config_template)
        preds = walk_forward_train_predict(
            wide_returns, config,
            train_end=TRAIN_END,
            test_end=wide_returns.index.max(),
        )
        walk_forward_predictions.append(preds)

    walk_forward_predictions = pd.concat(walk_forward_predictions, ignore_index=True)
    walk_forward_predictions.to_parquet(
        DATA_PROCESSED / "lstm_gru_predictions_walkforward.parquet",
        index=False,
    )
    print("Saved:", walk_forward_predictions.shape)
else:
    print("Walk-forward DL retraining is off. Flip QUICK_MODE to False above, "
          "flip RUN_WALK_FORWARD_DL to True here, and run this locally — "
          "budget real time (see markdown above).")